In [1]:
import tensorflow as tf # type: ignore
import pathlib
import matplotlib.pyplot as plt
import os
import numpy as np
from dotenv import load_dotenv


### 1. Configurar o caminho e criar lista de imagens ###
load_dotenv()
print(f"Arquivo .env carregado: {load_dotenv()}")

img_dir_str = os.getenv("BASE_IMG_FOLDER")
print(f"BASE_IMG_FOLDER: {os.getenv('BASE_IMG_FOLDER')}")
if img_dir_str is None:
    raise ValueError("A variável de ambiente BASE_IMG_FOLDER não está definida no arquivo .env")
img_dir = pathlib.Path(img_dir_str)

mask_dir_str = os.getenv("NUMPY_FOLDER")
print(f"NUMPY_FOLDER: {os.getenv('NUMPY_FOLDER')}")
if mask_dir_str is None:
    raise ValueError("A variável de ambiente NUMPY_FOLDER não está definida no arquivo .env")
mask_dir = pathlib.Path(mask_dir_str)

# Verificar se o diretório existe
if not os.path.exists(mask_dir):
    raise ValueError(f"Diretório não encontrado: {mask_dir}")

# Função para extrair o número do nome do arquivo
def get_file_number(filepath):
    return int(os.path.basename(filepath).replace('.png', ''))
def get_file_number_mask(filepath):
    return int(os.path.basename(filepath).replace('.npy', ''))

# Verificar se os arquivos existem e converter PosixPath para strings 
mask_files = []
for path in mask_dir.glob('*.npy'):
    if os.path.exists(path):
        mask_files.append(str(path))

image_files = []
for path in img_dir.glob('*.png'):
    if os.path.exists(path):
        image_files.append(str(path))

# Ordenar a lista usando o número do arquivo como chave
image_files = sorted(image_files, key=get_file_number)
mask_files = sorted(mask_files, key=get_file_number_mask)

print(f"Número de arquivos de imagem encontrados: {len(image_files)}")
print(f"Número de arquivos de máscara encontrados: {len(mask_files)}")

# Mostrar alguns exemplos de nomes de arquivos
print("Exemplos de nomes de arquivos de imagem:")
for i in image_files[:5]:
    print(i)
    
print("Exemplos de nomes de arquivos de máscara:")
for i in mask_files[:5]:
    print(i)

if len(image_files) != len(mask_files):
    raise ValueError("O número de imagens e máscaras não corresponde")

mask_count = len(mask_files)
print(f"Encontradas {mask_count} máscaras")

# Verificar se temos imagens
if mask_count == 0:
    raise ValueError(f"Nenhuma imagem PNG encontrada em: {mask_dir}")

# Mostrar alguns caminhos de exemplo
print("\nPrimeiros 5 caminhos de máscaras:")
for path in mask_files[:5]:
    print(path)

In [ ]:
import numpy as np
# Linux
mask = np.load(r'/media/renato/Data/PIBIT/NumpyFiles/2750.npy')

# Windows
# mask = np.load('C:\\Users\\Renato\\Documents\\PIBIT\\NumpyFiles\\000.npy')

In [ ]:
mask.shape

In [ ]:
plt.imshow(mask[:,:,6], cmap='gray')

In [4]:
def load_and_preprocess_image_mask(image_path, mask_path):
    try:
        # Carregar imagem RGB
        img = tf.io.read_file(image_path)
        img = tf.image.decode_png(img, channels=3)
        img = tf.image.resize(img, [512, 512])  # Redimensionar o tamanho
        img = tf.cast(img, tf.float32) / 255.0  # Normalizar a imagem

        # Carregar máscara do arquivo numpy
        mask = np.load(mask_path.numpy().decode())  # Ler a máscara
        mask = tf.convert_to_tensor(mask, dtype=tf.float32)  # Converter como tensor

        # Normalizar máscara
        mask = mask / 255.0  # Converter valores de 0-255 a 0-1
        mask = tf.where(mask >= 0.3, 1.0, 0.0)  # Binarizar a máscara

        return img, mask

    except Exception as e:
        print(f"Erro ao processar imagem/máscara {image_path}, {mask_path}: {str(e)}")
        raise


## Função para aplicar aumento de dados considerando máscaras como arrays multicamadas
def augment_image_mask(image, mask):
    # Gere um valor aleatório comum para transformações
    flip_left_right = tf.random.uniform(shape=[]) > 0.5
    flip_up_down = tf.random.uniform(shape=[]) > 0.5
    k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)  # 0, 90, 180, 270 graus

    # Flip horizontal
    if flip_left_right:
        image = tf.image.flip_left_right(image)
        mask = tf.image.flip_left_right(mask)

    # Flip vertical
    if flip_up_down:
        image = tf.image.flip_up_down(image)
        mask = tf.image.flip_up_down(mask)

    # Aplicar rotação à imagem
    image = tf.image.rot90(image, k=k)

    # Aplique rotação à máscara em toda a sua dimensão
    mask = tf.image.rot90(mask, k=k)  # Agora giramos a máscara inteira de uma vez

    # Configurações adicionais de imagem
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, 0.8, 1.2)

    return image, mask

# Converter listas de rotas em conjunto de dados do TensorFlow
def process_path(image_path, mask_path):
    img, mask = tf.py_function(load_and_preprocess_image_mask, [image_path, mask_path], [tf.float32, tf.float32])
    img.set_shape((512, 512, 3))  # Defina o tamanho esperado da imagem
    mask.set_shape((512, 512, 22))  # Defina o tamanho esperado da máscara
    return img, mask

# Criar um conjunto de dados a partir de listas de imagens e máscaras
train_dataset = tf.data.Dataset.from_tensor_slices((image_files, mask_files))
train_dataset = train_dataset.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)

# Aplicar Aumento de Dados
train_dataset = train_dataset.map(lambda img, mask: augment_image_mask(img, mask), num_parallel_calls=tf.data.AUTOTUNE)

"""
O BATCH_SIZE determina quantas imagens e máscaras são processadas simultaneamente pela GPU/CPU durante o treinamento.
O BUFFER_SIZE determina quantos elementos são armazenados em memória para embaralhamento eficiente.
"""

### Configurar o conjunto de dados ###
# TF-CPU: Configuração original
# BUFFER_SIZE = 100
# BATCH_SIZE = 32
# TF-GPU: Configuração priorizando velocidade [Estouro de memória em treinamento]
# BUFFER_SIZE = 128
# BATCH_SIZE = 32
# TF-GPU: Configuração balanceada para performance e uso de memória [Estouro de memória em treinamento]
# BUFFER_SIZE = 64
# BATCH_SIZE = 16
# TF-GPU: Configuração para economia de memória [Sucesso usando o limite máximo de recursos do meu computador]
BUFFER_SIZE = 32
BATCH_SIZE = 8

train_dataset = (
    train_dataset
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# Inspecionar o conjunto de dados
for img, mask in train_dataset.take(1):
    print(f"Imagem em lote: {img.shape}")  # Esperado: (batch_size, 512, 512, 3)
    print(f"Máscara em lote: {mask.shape}")  # Esperado: (batch_size, 512, 512, 22)

In [5]:
mask[0]

In [6]:
def unet_multilabel_v3(input_shape=(512, 512, 3), num_classes=22):
    # Input
    inputs = tf.keras.layers.Input(shape=input_shape)
    
    # Encoder (Downsampling)
    def encoder_block(x, filters, kernel_size=(3, 3), padding="same", dropout_rate=0.1):
        x = tf.keras.layers.Conv2D(filters, kernel_size, padding=padding, activation=None)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation("relu")(x)
        x = tf.keras.layers.Conv2D(filters, kernel_size, padding=padding, activation=None)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation("relu")(x)
        x = tf.keras.layers.Dropout(dropout_rate)(x)
        return x

    # Encoder - mantendo a estrutura original de 3 níveis, mas com separação entre normalização e ativação
    e1 = encoder_block(inputs, 64)
    p1 = tf.keras.layers.MaxPooling2D((2, 2))(e1)
    
    e2 = encoder_block(p1, 128)
    p2 = tf.keras.layers.MaxPooling2D((2, 2))(e2)
    
    e3 = encoder_block(p2, 256)
    p3 = tf.keras.layers.MaxPooling2D((2, 2))(e3)
    
    # Bottleneck - com um número moderado de filtros
    bottleneck = encoder_block(p3, 512, dropout_rate=0.3)
    
    # Decoder (Upsampling) - estrutura aprimorada mas sem camadas extra
    def decoder_block(x, skip_features, filters, kernel_size=(3, 3), padding="same", dropout_rate=0.1):
        # Upsampling
        x = tf.keras.layers.UpSampling2D((2, 2))(x)
        
        # Camada de convolução após upsampling
        x = tf.keras.layers.Conv2D(filters, (2, 2), padding="same", activation=None)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation("relu")(x)
        
        # Concatenate com skip connection
        x = tf.keras.layers.Concatenate()([x, skip_features])
        
        # Convolução com normalização e ativação separadas
        x = tf.keras.layers.Conv2D(filters, kernel_size, padding=padding, activation=None)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation("relu")(x)
        
        x = tf.keras.layers.Conv2D(filters, kernel_size, padding=padding, activation=None)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation("relu")(x)
        
        x = tf.keras.layers.Dropout(dropout_rate)(x)
        
        return x
    
    # Decoder - mantendo estrutura original de 3 níveis
    d3 = decoder_block(bottleneck, e3, 256, dropout_rate=0.1)
    d2 = decoder_block(d3, e2, 128, dropout_rate=0.1)
    d1 = decoder_block(d2, e1, 64, dropout_rate=0.1)
    
    # Mecanismo de atenção simplificado
    attention_gate = tf.keras.layers.Conv2D(1, (1, 1), padding="same", activation="sigmoid")(d1)
    attention_features = tf.keras.layers.Multiply()([d1, attention_gate])
    
    # Output com sigmoid activation para multilabel binary masks
    outputs = tf.keras.layers.Conv2D(num_classes, (1, 1), activation="sigmoid")(attention_features)
    
    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    return model


# Create model
model = unet_multilabel_v3()
model.summary()

In [7]:
### Amostras
# Obter o número total de amostras no conjunto de dados sem lote prévio
dataset = tf.data.Dataset.from_tensor_slices((image_files, mask_files))
dataset = dataset.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
dataset_size = tf.data.experimental.cardinality(dataset).numpy()

# Definir proporções para treinar, validar e testar
train_size = int(0.8 * dataset_size)
val_size = int(0.1 * dataset_size)
test_size = dataset_size - train_size - val_size

# Mesclar o conjunto de dados antes de dividir
shuffled_dataset = dataset.shuffle(BUFFER_SIZE)
shuffled_dataset = shuffled_dataset.shuffle(BUFFER_SIZE)

# Divida o conjunto de dados ANTES de aplicar o lote para evitar o problema de lote duplo
train_dataset = shuffled_dataset.take(train_size)
remaining_dataset = shuffled_dataset.skip(train_size)
val_dataset = remaining_dataset.take(val_size)
test_dataset = remaining_dataset.skip(val_size)

# Aplicar aumento de dados apenas ao conjunto de treinamento
train_dataset = train_dataset.map(lambda img, mask: augment_image_mask(img, mask), num_parallel_calls=tf.data.AUTOTUNE)

# Aplicar lote após divisão
# train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
# val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
# test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(1)
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(1)
test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(1)

"""
O valor de prefetch igual a 1 significa que enquanto o modelo processa um batch, 
o pipeline carrega apenas o próximo batch, minimizando o uso de memória.
Isto deve resolver o problema de alocação de memória excessiva.
"""

# Confirmar os tamanhos do conjunto
print(f'Tamanho do conjunto de dados de treinamento: {train_size}')
print(f'Tamanho do conjunto de dados de validação: {val_size}')
print(f'Tamanho do conjunto de dados de avaliação: {test_size}')


In [8]:
import gc

# Limpar memória antes do treinamento
gc.collect()
tf.keras.backend.clear_session()

In [9]:
# Redefinição da Perda
# Useremos BCE + Dice Loss ao invés de Focal Loss

def dice_coef(y_true, y_pred, smooth=1e-6):
    """
    Calcula o coeficiente Dice entre as máscaras verdadeiras e preditas
    """
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    """
    Calcula o Dice Loss (1 - coeficiente Dice)
    """
    return 1 - dice_coef(y_true, y_pred)

def multilabel_dice_loss(y_true, y_pred):
    """
    Aplica Dice Loss para cada canal (classe) e calcula a média
    """
    loss = 0
    for i in range(y_pred.shape[-1]):  # Iterar por cada classe
        loss += dice_loss(y_true[..., i], y_pred[..., i])
    
    # Retorna a média das perdas em todas as classes
    return loss / y_pred.shape[-1]

def bce_dice_loss(y_true, y_pred, dice_weight=0.5):
    """
    Combinação de BCE e Dice Loss
    
    Parâmetros:
    - dice_weight: Peso dado ao Dice Loss (0 a 1) 0 = apenas BCE, 1 = apenas Dice

    Considerações:
    Ajuste do peso do Dice Loss.
    Inicialmente, estamos usando um equilíbrio 50/50 entre BCE e Dice Loss (dice_weight=0.5). 
    Dependendo dos resultados, pode ser necessário ajustar este peso para dar mais importância a um ou outro componente:
        Para focar mais na qualidade das bordas: aumente o peso para dice_weight=0.7;
        Para focar mais na classificação a nível de pixel: reduza para dice_weight=0.3.
    """
    bce = tf.keras.losses.BinaryCrossentropy(from_logits=False)(y_true, y_pred)
    dice = multilabel_dice_loss(y_true, y_pred)
    
    return (1 - dice_weight) * bce + dice_weight * dice

def loss_function(y_true, y_pred):
    return bce_dice_loss(y_true, y_pred, dice_weight=0.5)

In [10]:
# Redefinição da métrica IoU
"""
Problema:
* Métrica IoU (Intersection over Union) está fixa em 0.4774 durante todo o treinamento

Possíveis causas:
* Incompatibilidade com segmentação multicamada: A implementação padrão MeanIoU do TensorFlow é projetada para segmentação de classe única
(onde cada pixel pertence a exatamente uma classe). No seu caso, você está fazendo uma segmentação multicamada onde um pixel pode pertencer a 
várias classes.

* Problema de formato dos dados: A MeanIoU espera entradas de forma específica, e suas máscaras de 22 canais podem não estar no formato esperado.

* Problemas com a ativação sigmoid: Como você está usando sigmoid na camada de saída, suas previsões são probabilidades entre 0 e 1, 
mas a métrica IoU espera valores discretos (classes).

Solução:
* Implementar uma versão personalizada da métrica IoU que funcione corretamente com segmentação multicamada e sigmoid.
* Implementação nas duas fases.
"""

class MulticlassIoU(tf.keras.metrics.Metric):
    def __init__(self, num_classes=22, threshold=0.5, name='multiclass_iou', **kwargs):
        super(MulticlassIoU, self).__init__(name=name, **kwargs)
        self.num_classes = num_classes
        self.threshold = threshold
        self.total_iou = self.add_weight(name='total_iou', initializer='zeros')
        self.count = self.add_weight(name='count', initializer='zeros')
        
    def update_state(self, y_true, y_pred, sample_weight=None):
        # Binarize predictions
        y_pred = tf.cast(y_pred > self.threshold, tf.float32)
        
        # Calculate intersection and union for all classes at once
        intersection = tf.reduce_sum(y_true * y_pred, axis=[0, 1, 2])
        union = tf.reduce_sum(y_true, axis=[0, 1, 2]) + tf.reduce_sum(y_pred, axis=[0, 1, 2]) - intersection
        
        # Calculate IoU for each class
        iou = tf.where(union > 0, intersection / (union + 1e-10), tf.ones_like(intersection))
        
        # Calculate mean IoU across all classes
        mean_iou = tf.reduce_mean(iou)
        
        # Update metric state
        self.total_iou.assign_add(mean_iou)
        self.count.assign_add(1.0)
        
    def result(self):
        return self.total_iou / (self.count + 1e-10)
    
    def reset_state(self):
        self.total_iou.assign(0.0)
        self.count.assign(0.0)

In [ ]:
### Teste rápido da métrica
for img, mask in train_dataset.take(1):
    # Fazer uma previsão
    pred = model.predict(img)
    
    # Calcular IoU manualmente
    iou_metric = MulticlassIoU(num_classes=22)
    iou_metric.update_state(mask, pred)
    print(f"IoU calculado: {iou_metric.result().numpy()}")
    
    # Verificar se varia com diferentes limiares
    for threshold in [0.3, 0.5, 0.7]:
        iou_metric = MulticlassIoU(num_classes=22, threshold=threshold)
        iou_metric.update_state(mask, pred)
        print(f"IoU com limiar {threshold}: {iou_metric.result().numpy()}")

"""
Resultados:
IoU calculado: 0.03851988911628723
IoU com limiar 0.3: 0.04670115187764168
IoU com limiar 0.5: 0.03851988911628723
IoU com limiar 0.7: 0.5
"""

"""
Nova métrica funciona corretamente.
* O valor de 0.5 (50%) obtido com limiar 0.7 é suspeito. Isso provavelmente significa que com um limiar tão alto, quase nenhum pixel é classificado como 
positivo, tanto nas previsões quanto nas máscaras verdadeiras, resultando em poucos falsos positivos e falsos negativos, mas também poucas detecções reais. 
Isso pode produzir um IoU artificialmente alto. É necessário revisar o modelo v2.

* Os valores baixos (3.9% e 4.7%) com limiares 0.5 e 0.3 são mais realistas e explicam melhor a discrepância visual que você estava observando entre as máscaras 
reais e previstas. 

* A métrica está funcionando corretamente, pois está respondendo às mudanças de limiar e não está mais presa no valor constante 0.4774.
"""

In [11]:
# Callbacks para melhorar o treinamento
callbacks = [
    # Early stopping para evitar Overfitting (sobreajuste)
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,  # Permitir mais épocas antes de se deter se não houver melhor
        restore_best_weights=True,
        verbose=1
    ),

    # Salvar o melhor modelo baseado na menor perda de validação
    tf.keras.callbacks.ModelCheckpoint(
        # filepath='best_model_focal_v2.keras',     # model_v2
        filepath='best_model_bce_dice_v3.keras',    # model_v3
        save_best_only=True,
        monitor='val_loss',
        mode='min',
        verbose=1
    ),

    # Reduza a taxa de aprendizagem se a perda de validação persistir
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,  # Reduza a tarefa de aprendizagem para a metade se não for melhor
        patience=5,  # Esperar 5 épocas sem melhorar antes de reduzir
        min_lr=1e-6,  # Limite inferior para a tarefa de aprendizagem
        verbose=1
    ),

    # Registrador de métricas no TensorBoard
    tf.keras.callbacks.TensorBoard(
        # log_dir='logs_focal',     # model_v2
        log_dir='logs_bce_dice_v3', # model_v3
        histogram_freq=1,
        write_graph=True,
        write_images=True
    )
]

# Compilar o modelo com perda focal e especificações específicas
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),  # Taxa de aprendizagem inicial
    loss=loss_function,
    metrics=[
        'accuracy', 
        MulticlassIoU(num_classes=22, threshold=0.5), # Implementação da métrica para multiclasse
        tf.keras.metrics.Precision(), 
        tf.keras.metrics.Recall()
    ]
)

# Definir a estratégia de treinamento em fases (ajuste fino)
EPOCHS_PHASE_1 = 75  # Treinamento inicial
EPOCHS_PHASE_2 = 25  # Ajuste fino

print("Treinamento: Fase 1 - Aprendizagem inicial")
history_1 = model.fit(
    train_dataset,
    epochs=EPOCHS_PHASE_1,
    validation_data=val_dataset,
    callbacks=callbacks,
    verbose=1
)

# Recompilar o modelo para a fase de ajuste fino com taxa de aprendizagem reduzida
print("Recompilando modelo para ajuste fino...")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # Taxa reduzida para ajuste fino
    loss=loss_function,
    metrics=[
        'accuracy', 
        MulticlassIoU(num_classes=22, threshold=0.5), # Implementação da métrica para multiclasse
        tf.keras.metrics.Precision(), 
        tf.keras.metrics.Recall()
    ]
)

print("Treinamento: Fase 2 - Ajuste fino com taxa de aprendizagem reduzida")
history_2 = model.fit(
    train_dataset,
    epochs=EPOCHS_PHASE_2,
    validation_data=val_dataset,
    callbacks=callbacks,
    verbose=1
)


In [ ]:
# Visualização da evolução do treinamento
import matplotlib.pyplot as plt

def plot_training_metrics(history_1, history_2):
    print("Chaves disponíveis em history_1:", list(history_1.history.keys()))
    print("Chaves disponíveis em history_2:", list(history_2.history.keys()))

    combined_history = {
        'loss': history_1.history['loss'] + history_2.history['loss'],
        'val_loss': history_1.history['val_loss'] + history_2.history['val_loss'],
        'accuracy': history_1.history['accuracy'] + history_2.history['accuracy'],
        'val_accuracy': history_1.history['val_accuracy'] + history_2.history['val_accuracy'],
        # Usando os nomes corretos das métricas para cada histórico
        'precision': history_1.history['precision'] + history_2.history['precision_1'],
        'val_precision': history_1.history['val_precision'] + history_2.history['val_precision_1'],
        'recall': history_1.history['recall'] + history_2.history['recall_1'],
        'val_recall': history_1.history['val_recall'] + history_2.history['val_recall_1'],
    }

    plt.figure(figsize=(15, 5))
    
    # Perda
    plt.subplot(1, 3, 1)
    plt.plot(combined_history['loss'], label='Perda de treinamento')
    plt.plot(combined_history['val_loss'], label='Perda de validação')
    plt.title('Evolução da perda')
    plt.legend()

    # Precisão
    plt.subplot(1, 3, 2)
    plt.plot(combined_history['accuracy'], label='Precisão de treinamento')
    plt.plot(combined_history['val_accuracy'], label='Precisão de validação')
    plt.title('Evolução da Precisão')
    plt.legend()

    # Recall
    plt.subplot(1, 3, 3)
    plt.plot(combined_history['recall'], label='Recall de treinamento')
    plt.plot(combined_history['val_recall'], label='Recall de validação')
    plt.title('Evolução do Recall')
    plt.legend()

    plt.tight_layout()
    plt.show()


plot_training_metrics(history_1, history_2)

In [ ]:
def focal_loss(alpha=0.25, gamma=2.0):
    def loss(y_true, y_pred):
        y_pred = tf.keras.backend.clip(y_pred, 1e-7, 1 - 1e-7)
        focal_loss = -alpha * y_true * tf.keras.backend.pow(1 - y_pred, gamma) * tf.keras.backend.log(y_pred)
        focal_loss -= (1 - alpha) * (1 - y_true) * tf.keras.backend.pow(y_pred, gamma) * tf.keras.backend.log(1 - y_pred)
        return tf.keras.backend.mean(focal_loss)
    return loss

In [ ]:
# Carregar o modelo salvo
model = tf.keras.models.load_model('best_model_focal.keras', custom_objects={'loss': focal_loss()})

In [ ]:
import matplotlib.pyplot as plt

def visualize_multilabel_predictions(model, dataset):
    for img, mask in dataset.take(1):
        pred_mask = model.predict(tf.expand_dims(img[0], axis=0))[0]

        plt.figure(figsize=(10, 5))

        # Imagem original
        plt.subplot(1, 3, 1)
        plt.imshow(img[0].numpy())
        plt.title("Imagem Original")

        # Ver classe 0 real e prevista
        plt.subplot(1, 3, 2)
        plt.imshow(mask[0][:, :, 0], cmap='gray')
        plt.title("Máscara Real (Classe 10)")

        plt.subplot(1, 3, 3)
        plt.imshow(pred_mask[:, :, 0], cmap='gray')
        plt.title("Máscara Prevista (Classe 10)")

        plt.show()
        break

visualize_multilabel_predictions(model, train_dataset)